In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# apoe_df = pd.read_csv('~/BCG/biomarkers/data/APOE_270125.csv')
apoe_df = pd.read_excel('~/BCG/biomarkers/data/APOE.xlsx')
print(apoe_df.shape)
apoe_df.columns = ['#', 'Genotype']
apoe_df.head()

In [ ]:
# drop rows with the same ID (#) values 
apoe_df = apoe_df.drop_duplicates(subset='#', keep='first')

In [ ]:
def numberize_apoe(apoe):
    if apoe == 'APOE4/4':
        return 2
    elif apoe == 'APOE3/4' or apoe == 'APO3/3':
        return 1
    elif apoe == 'APOE3/3' or apoe == 'APO3/4':
        return 0
    else:
        return None
    
def edit_id(id_num):
    if 'PB' in id_num:
        return  'P(B)' + id_num[2:]
    elif 'P' in id_num:
        return  'P(B)' + id_num[1:]
    else:
        return id_num

In [ ]:
apoe_df['Genotype'].value_counts()

In [ ]:
healthy_color = '#b4cde3' # blue
risk_color = '#fbb4ae' # red
inter_color = '#fed9a6'

graph_colors = [healthy_color, inter_color, risk_color]
explode = (0.1, 0.1, 0.1)
n_apoe_33, n_apoe_34, n_apoe_44 = apoe_df['Genotype'].apply(numberize_apoe).value_counts()

plt.pie(apoe_df['Genotype'].apply(numberize_apoe).value_counts(), autopct='%1.1f%%', colors=graph_colors, shadow=True, explode=explode)
plt.axis('equal')
plt.legend(loc='upper right', labels=[f'ApoE3\\3 N={n_apoe_33}', f'ApoE3\\4 N={n_apoe_34}', f'ApoE4\\4 N={n_apoe_44}'])
plt.title('APOE Genotype Distribution')

In [ ]:
apoe_df['APOE'] = apoe_df['Genotype'].apply(numberize_apoe)
apoe_df['#'] = apoe_df['#'].apply(edit_id)

In [ ]:
apoe_df['APOE'].value_counts()

In [ ]:
# apoe_df.to_csv('~/BCG/biomarkers/data/APOE_250625.csv', index=False)

In [ ]:
data_path = '/ems/elsc-labs/habib-n/yuval.rom/BCG/clinical_data/Data/fightAD_general_data_table.xlsx'
df = pd.read_excel(data_path, sheet_name='1')
df.columns = df.loc[0]
df = df.drop(0)
df = df.iloc[:,:-2]
print(df.shape)
df.head()

In [ ]:
(~df['APOE-SNP'].isna()).sum()

In [ ]:
df1 = df.merge(apoe_df, how='left', left_on='#', right_on='#')
print(df.shape[0] == df1.shape[0])
df1.index = df.index

In [ ]:
df1.columns

In [ ]:
(df1.drop(columns=['APOE', 'Genotype'], inplace=False).index == df.index).sum()

In [ ]:
df1.index, df.index

In [ ]:
df.columns[(df1.drop(columns=['APOE', 'Genotype'], inplace=False) == df).all()]

In [ ]:
df.shape

In [ ]:
apoe_df['#'][(~apoe_df['#'].isin(df['#']))]

In [ ]:
(~apoe_df['#'].isin(df['#'])).sum()

In [ ]:
df1.APOE.value_counts()

In [ ]:
# 1. Prepare the data
df_hist = df1[['P-tau217', 'APOE']].dropna()
df_hist['APOE'] = df_hist['APOE'].astype(int)  # Ensure integer for mapping

# 2. Define colors
apoe_colors = {0: '#b4cde3', 1: '#fed9a6', 2: '#fbb4ae'}
color_list = [apoe_colors[i] for i in sorted(apoe_colors.keys())]

# 3. Prepare data for stacking
apoe_groups = [df_hist[df_hist['APOE'] == i]['P-tau217'] for i in sorted(apoe_colors.keys())]

# 4. Plot
plt.figure(figsize=(8,6))
plt.hist(apoe_groups, bins=50, stacked=True, color=color_list, label=['ApoE3/3', 'ApoE3/4', 'ApoE4/4'])
plt.xlabel('P-tau217')
plt.ylabel('Count')
plt.title('Histogram of binned P-tau217, stacked by APOE')
plt.legend(title='APOE Genotype')
plt.show()

In [ ]:
# Ensure no missing values
df_corr = df1[['P-tau217', 'APOE']].dropna()
df_corr['APOE'] = df_corr['APOE'].astype(int)

# Calculate Pearson correlation
correlation = df_corr['P-tau217'].corr(df_corr['APOE'])
print(f"Pearson correlation between APOE and P-tau217: {correlation:.3f}")

In [ ]:
from scipy.stats import spearmanr

spearman_corr, p_value = spearmanr(df_corr['P-tau217'], df_corr['APOE'])
print(f"Spearman correlation: {spearman_corr:.3f}, p-value: {p_value:.3g}")